In [ ]:
# Auto-injected: export every displayed figure as PNG + PDF + SVG into png/ pdf/ svg/
import os
import matplotlib.pyplot as plt
_PDF_STEM = 'fig2_1'
_PDF_N = [0]
for _d in ('pdf', 'svg', 'png'):
    os.makedirs(_d, exist_ok=True)
def _save_fig(base, fig=None, dpi=300, **kw):
    f = fig if fig is not None else plt.gcf()
    f.savefig(f'pdf/{base}.pdf', bbox_inches='tight', **kw)
    f.savefig(f'svg/{base}.svg', bbox_inches='tight', **kw)
    f.savefig(f'png/{base}.png', dpi=dpi, bbox_inches='tight', **kw)
def _save_pdf():
    _PDF_N[0] += 1
    _save_fig(f'{_PDF_STEM}_{_PDF_N[0]:02d}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from pathlib import Path

# 设置字体为Times New Roman
plt.rcParams['font.family'] = 'Arial'

# 读取可行性分析结果数据
data_path = Path("../result/island_viability_summary_electric.csv")
df = pd.read_csv(data_path)


In [ ]:
# 导入必要的库
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import pandas as pd
import numpy as np
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy import stats
from matplotlib.patches import Rectangle
from pathlib import Path # 引入 Path 用于处理路径
import matplotlib.colors

# --- Nature 风格图表设置 ---
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 20,
    'axes.labelsize': 18,
    'axes.titlesize': 16,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'figure.dpi': 300,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

# <<<< MODIFIED 1: 重新定义六个区域的名称和颜色 >>>>
# 这些名称直接反映了新的决策框架
SIX_REGION_NAMES = [
    "Feasible\nLow Cost\nHigh Affordability",       # 区域1: 可行, 低成本, 高支付能力
    "Feasible\nHigh Cost\nHigh Affordability",      # 区域2: 可行, 高成本, 高支付能力
    "Feasible\nLow Cost\nLow Affordability",        # 区域3: 可行, 低成本, 低支付能力
    "Infeasible\nHigh Cost\nHigh Affordability",    # 区域4: 不可行, 高成本, 高支付能力
    "Infeasible\nLow Cost\nLow Affordability",      # 区域5: 不可行, 低成本, 低支付能力
    "Infeasible\nHigh Cost\nLow Affordability",     # 区域6: 不可行, 高成本, 低支付能力
]
# 设计一套新的颜色方案，例如：蓝色系代表可行，红色系代表不可行
SIX_REGION_COLORS = {
    SIX_REGION_NAMES[0]: '#012A61', # 深蓝 (最理想)
    SIX_REGION_NAMES[1]: '#0B75B3', # 中蓝
    SIX_REGION_NAMES[2]: '#89CAEA', # 浅蓝 (勉强可行)
    SIX_REGION_NAMES[3]: "#F0D2D2", # 浅红 (成本问题)
    SIX_REGION_NAMES[4]: '#DC5654', # 中红 (支付能力问题)
    SIX_REGION_NAMES[5]: '#982B2D', # 深红 (最棘手)
}

# --- 全局中位线定义 ---
# <<<< MODIFIED 2: 移除可负担能力中位数 >>>>
GLOBAL_MEDIAN_BREAKEVEN = None

def initialize_global_medians(df):
    global GLOBAL_MEDIAN_BREAKEVEN
    df_baseline = df[df['scenario'] == 'output_0'].copy()
    if not df_baseline.empty:
        GLOBAL_MEDIAN_BREAKEVEN = df_baseline['tariff_breakeven'].median()
        print(f"全局成本中位线已设置（基于output_0场景）：")
        print(f"   - 成本回收电价中位数: {GLOBAL_MEDIAN_BREAKEVEN:.3f} USD/kWh")
    else:
        print("警告：未找到output_0场景数据，无法设置全局中位线！")

def assign_ipcc_region(lat, lon, ipcc_regions_gdf):
    point = Point(lon, lat)
    possible_matches_index = list(ipcc_regions_gdf.sindex.intersection(point.bounds))
    possible_matches = ipcc_regions_gdf.iloc[possible_matches_index]
    precise_matches = possible_matches[possible_matches.contains(point)]
    if not precise_matches.empty:
        return precise_matches.iloc[0]['Acronym']
    return 'Unknown'

# <<<< MODIFIED 3: 重写核心分类函数，以实现六区域逻辑 >>>>
def classify_point(x, y, median_breakeven):
    """
    对单个坐标点 (x, y) 进行分类，返回其所属的六个区域之一的名称。
    这里的核心是 affordability 的高低是与 median_breakeven 比较的。
    """
    is_low_cost = x <= median_breakeven
    is_high_affordability_benchmark = y > median_breakeven
    is_feasible = y >= x

    if is_feasible:
        if is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[0]  # Feasible, Low Cost, High Affordability
        elif not is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[1]  # Feasible, High Cost, High Affordability
        elif is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[2]  # Feasible, Low Cost, Low Affordability
        else: # not is_low_cost and not is_high_affordability_benchmark
            # 这个区域 (可行, 高成本, 低支付能力) 几乎不可能存在, 如果存在, 归为最棘手类
            return SIX_REGION_NAMES[5]
    else: # Infeasible
        if not is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[3]  # Infeasible, High Cost, High Affordability
        elif is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[4]  # Infeasible, Low Cost, Low Affordability
        elif not is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[5]  # Infeasible, High Cost, Low Affordability
        else: # is_low_cost and is_high_affordability_benchmark
             # 这个区域 (不可行, 低成本, 高支付能力) 逻辑上也不太可能存在, 归为成本问题类
            return SIX_REGION_NAMES[3]


def classify_regions_dataframe(df_scenario, median_breakeven):
    """
    对 DataFrame 中的每一行数据进行分类。
    """
    df_scenario['region'] = df_scenario.apply(
        lambda row: classify_point(row['tariff_breakeven'], row['tariff_affordable'], median_breakeven),
        axis=1
    )
    return df_scenario

# <<<< MODIFIED 4: 背景绘制函数适配六区域模型 >>>>
def draw_six_region_backgrounds(ax, median_breakeven, xlim, ylim):
    """
    使用网格化和 contourf 的方法精确绘制六个区域的背景色。
    """
    x_min, x_max = xlim
    y_min, y_max = ylim
    
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 500),
        np.linspace(y_min, y_max, 500)
    )
    
    region_to_id = {name: i for i, name in enumerate(SIX_REGION_NAMES)}
    
    vectorized_classify = np.vectorize(classify_point)
    # 注意这里只传递 median_breakeven
    grid_regions = vectorized_classify(xx, yy, median_breakeven)
    
    Z = np.zeros(grid_regions.shape)
    for name, region_id in region_to_id.items():
        Z[grid_regions == name] = region_id
        
    colors = [SIX_REGION_COLORS[name] for name in SIX_REGION_NAMES]
    cmap = matplotlib.colors.ListedColormap(colors)
    
    levels = np.arange(-0.5, len(colors), 1)
    ax.contourf(xx, yy, Z, levels=levels, cmap=cmap, alpha=0.6, zorder=0)

# --- 核心绘图函数 ---
# <<<< MODIFIED 5: 主绘图函数适配六区域模型 >>>>
def plot_feasibility_quadrant_with_six_regions(df_plot, color_map, zoom_config, scenario_name, output_path=None):
    if df_plot.empty:
        print(f"情景 '{scenario_name}' 数据为空，跳过绘图。")
        return

    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

    median_breakeven = GLOBAL_MEDIAN_BREAKEVEN
    
    x_min, x_max = -0.2, 2.5
    y_min, y_max = -0.2, 4.2
    
    ax.set_xlim(x_max, x_min)
    ax.set_ylim(y_min, y_max)
    
    # 调用新的六区域背景绘制函数
    draw_six_region_backgrounds(ax, median_breakeven, (x_min, x_max), (y_min, y_max))

    # --- 绘制主散点图 (逻辑不变) ---
    unique_regions = sorted(df_plot['ipcc_region'].unique())
    for region in unique_regions:
        region_data = df_plot[df_plot['ipcc_region'] == region]
        ax.scatter(region_data['tariff_breakeven'], region_data['tariff_affordable'],
                   color=color_map.get(region, 'gray'),
                   label=region,
                   alpha=0.8, s=30, edgecolors='white', linewidth=0.5, zorder=3)

    # --- 设置坐标轴和标题 ---
    ax.set_xlabel('Cost Recovery Electricity Price (USD kWh$^{-1}$)', labelpad=18)
    ax.set_ylabel('Affordable Electricity Price (USD kWh$^{-1}$)', labelpad=18)
    ax.spines[['right', 'top']].set_visible(False)

    # --- 添加辅助线 ---
    ax.axvline(x=median_breakeven, color='black', linestyle='--', linewidth=1.2, zorder=5)
    # 水平线现在也使用 median_breakeven
    ax.axhline(y=median_breakeven, color='black', linestyle='--', linewidth=1.2, zorder=5)
    
    lim_min = min(ax.get_xlim()[1], ax.get_ylim()[0])
    lim_max = max(ax.get_xlim()[0], ax.get_ylim()[1])
    ax.plot([lim_max, lim_min], [lim_max, lim_min], color='#be1420', linestyle='--', 
              alpha=1, linewidth=2, label='Break-even line', zorder=5)
    
    ax.grid(False)

    # --- 创建放大视图 ---
    zoom_data = df_plot[
        (df_plot['tariff_breakeven'].between(zoom_config['x_min'], zoom_config['x_max'])) &
        (df_plot['tariff_affordable'].between(zoom_config['y_min'], zoom_config['y_max']))
    ]
    
    if not zoom_data.empty:
        inset_ax = ax.inset_axes([0.1, 0.35, 0.65, 0.65])
        divider = make_axes_locatable(inset_ax)
        ax_top = divider.append_axes("top", size="25%", pad=0.05, sharex=inset_ax)
        ax_right = divider.append_axes("right", size="25%", pad=0.05, sharey=inset_ax)
        
        plt.setp(ax_top.get_xticklabels(), visible=False)
        plt.setp(ax_right.get_yticklabels(), visible=False)
        
        # 在子图中也绘制六区域背景
        draw_six_region_backgrounds(inset_ax, median_breakeven,
            (zoom_config['x_min'], zoom_config['x_max']),
            (zoom_config['y_min'], zoom_config['y_max']))
        
        for region in unique_regions:
            region_data = zoom_data[zoom_data['ipcc_region'] == region]
            if not region_data.empty:
                inset_ax.scatter(region_data['tariff_breakeven'], region_data['tariff_affordable'],
                                 color=color_map.get(region, 'gray'), alpha=0.8, s=40,
                                 edgecolors='white', linewidth=0.5)

        inset_ax.axvline(x=median_breakeven, color='black', linestyle='--', linewidth=1.2, zorder=1)
        inset_ax.axhline(y=median_breakeven, color='black', linestyle='--', linewidth=1.2, zorder=1)
        
        inset_lim_min = max(zoom_config['x_min'], zoom_config['y_min'])
        inset_lim_max = min(zoom_config['x_max'], zoom_config['y_max'])
        inset_ax.plot([inset_lim_max, inset_lim_min], [inset_lim_max, inset_lim_min],
                      color='#be1420', linestyle='--', alpha=1, linewidth=2, zorder=1)
        
        inset_ax.set_xlim(zoom_config['x_max'], zoom_config['x_min'])
        inset_ax.set_ylim(zoom_config['y_min'], zoom_config['y_max'])
        inset_ax.tick_params(labelsize=12)
        inset_ax.grid(False)

        # --- 添加密度曲线 (逻辑不变) ---
        x_data = zoom_data['tariff_breakeven']
        if len(x_data) > 1:
            kde_x = stats.gaussian_kde(x_data)
            x_range = np.linspace(zoom_config['x_min'], zoom_config['x_max'], 100)
            density_x = kde_x(x_range)
            ax_top.plot(x_range, density_x, color='#7a0101', linewidth=2)
            ax_top.fill_between(x_range, density_x, alpha=0.3, color='#a37070')
            ax_top.set_xlim(zoom_config['x_min'], zoom_config['x_max'])
            ax_top.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
            ax_top.set_yticks([])
            ax_top.set_facecolor('none')
            ax_top.invert_xaxis()
        
        y_data = zoom_data['tariff_affordable']
        if len(y_data) > 1:
            kde_y = stats.gaussian_kde(y_data)
            y_range = np.linspace(zoom_config['y_min'], zoom_config['y_max'], 100)
            density_y = kde_y(y_range)
            ax_right.plot(density_y, y_range, color="#A05901", linewidth=2)
            ax_right.fill_betweenx(y_range, density_y, alpha=0.3, color='#FF8C00')
            ax_right.set_ylim(zoom_config['y_min'], zoom_config['y_max'])
            ax_right.spines[['right', 'top', 'bottom', 'left']].set_visible(False)
            ax_right.set_xticks([])
            ax_right.set_facecolor('none')
        
        mark_inset(ax, inset_ax, loc1=1, loc2=4, fc="none", ec="0.5")

    fig.tight_layout()
    _save_pdf()
    plt.show()

# --- 统计和柱状图函数 ---
# <<<< MODIFIED 6: 统计函数适配六区域模型 >>>>
def analyze_and_plot_quadrant_statistics(df_scenario, scenario_name, median_breakeven):
    if df_scenario.empty:
        print(f"情景 '{scenario_name}' 数据为空，跳过柱状图。")
        return
        
    df_classified = classify_regions_dataframe(df_scenario, median_breakeven)
    counts = df_classified['region'].value_counts().to_dict()

    stats_df = pd.DataFrame(list(counts.items()), columns=['Category', 'Number of Islands'])
    
    category_order = SIX_REGION_NAMES
    color_palette = {name: SIX_REGION_COLORS[name] for name in category_order}

    fig, ax = plt.subplots(figsize=(6, 8), dpi=300)
    
    sns.barplot(
        data=stats_df,
        y='Category',
        x='Number of Islands',
        palette=color_palette,
        ax=ax,
        width=0.5,
        order=category_order,
        orient='h'
    )
    
    ax.set_ylabel('')
    ax.set_xlabel('Number of Islands', fontsize=20)
    ax.spines[['right', 'top']].set_visible(False)
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)

    for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.annotate(f'{int(width)}',
                        (width, p.get_y() + p.get_height() / 2.),
                        ha='left', va='center',
                        xytext=(5, 0),
                        textcoords='offset points',
                        fontsize=14)

    fig.tight_layout()
    _save_pdf()
    plt.show()
    
    total = sum(counts.values())
    print(f"\n--- 六区域统计分析结果 ({scenario_name}) ---")
    for category in category_order:
        count = counts.get(category, 0)
        percentage = (count / total * 100) if total > 0 else 0
        print(f"   {category.replace(chr(10), ' '):<55}: {count:4d} 个岛屿 ({percentage:.1f}%)")
    print("-" * 65)

# --- 图例生成函数 (保持不变) ---
def create_legend_only(color_map):
    fig_legend = plt.figure(figsize=(10, 4), dpi=300)
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w',
                                  markerfacecolor=color, markersize=12,
                                  label=region)
                       for region, color in color_map.items()]
    
    legend = fig_legend.legend(handles=legend_elements, loc='center', frameon=False,
                               title='IPCC Regions', title_fontsize=20,
                               fontsize=16, ncol=8)
    
    fig_legend.gca().set_axis_off()
    plt.tight_layout()
    _save_pdf()
    plt.show()
    
# ===== 主程序 =====
if __name__ == '__main__':
    # --- 1. 数据加载和预处理---
    ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")
    ipcc_regions.sindex
    print("正在为岛屿分配IPCC区域...")
    df['ipcc_region'] = df.apply(
        lambda row: assign_ipcc_region(row['lat'], row['lon'], ipcc_regions), axis=1
    )

    # --- 2. 初始化全局中位线 ---
    initialize_global_medians(df)

    # --- 3. 绘图配置 ---
    MIN_ISLANDS_PER_REGION = 0
    ZOOM_BOX = {'x_min': 0.08, 'x_max': 0.5, 'y_min': -0.03, 'y_max': 0.8}
    
    region_counts = df['ipcc_region'].value_counts()
    valid_regions = region_counts[region_counts > MIN_ISLANDS_PER_REGION].index.tolist()
    
    # 使用专业的调色板
    image_colors = [
        # 第1行
        '#0ddbf5', '#1d9bf7', '#8386fc', '#303cf9', '#fe5357', '#fd7c1a', '#ffbd15', '#fcff07',
        # 第2行
        '#444577', '#c65861', '#f3dee0', '#ffa725', '#ff6b62', '#be588d', '#58538b',
        # 第3行
        '#4292c9', '#a0c9e5', '#AA4499', '#88CCEE', '#f26a11', '#fe9376', '#817cb9', '#bcbddd',
        # 第4行
        '#cb78a6', '#d35f00', '#f7ec44', '#DDCC77', '#fcb93e', '#0072b2', '#979797',
        # 第5行
        '#b4403e', '#256ea2', '#ea841e', '#332288', '#603c87', '#9d5c39'
    ]
    region_color_map = {region: image_colors[i % len(image_colors)] for i, region in enumerate(valid_regions)}

    # --- 4. 循环处理并绘图 ---
    # <<<< MODIFIED 7: 更新主循环中的函数调用 >>>>
    scenarios = df['scenario'].unique()
    for scenario in scenarios:
        print(f"\n--- 正在处理情景: {scenario} ---")
        df_scenario = df[(df['scenario'] == scenario) & (df['ipcc_region'].isin(valid_regions))].copy()

        plot_feasibility_quadrant_with_six_regions( # <--- 函数名已更新
            df_plot=df_scenario,
            color_map=region_color_map,
            zoom_config=ZOOM_BOX,
            scenario_name=scenario
        )
        
        if not df_scenario.empty and GLOBAL_MEDIAN_BREAKEVEN is not None:
            analyze_and_plot_quadrant_statistics( # <--- 函数名已更新
                df_scenario, scenario, GLOBAL_MEDIAN_BREAKEVEN # <--- 参数已更新
            )

    # --- 5. 最后单独生成一次图例 ---
    print("\n--- 生成单独的图例 ---")
    create_legend_only(region_color_map)

In [ ]:
# 导入必要的库
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors

# --- Nature 风格图表设置 ---
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 14, # 调整字号以适应图例标签
    'axes.labelsize': 18,
    'axes.titlesize': 16,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'figure.dpi': 300,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

# <<<< COPIED 1: 从主代码复制六区域定义，确保一致 >>>>
SIX_REGION_NAMES = [
    "Feasible\nLow Cost\nHigh Affordability",       # 区域1
    "Feasible\nHigh Cost\nHigh Affordability",      # 区域2
    "Feasible\nLow Cost\nLow Affordability",        # 区域3
    "Infeasible\nHigh Cost\nHigh Affordability",    # 区域4
    "Infeasible\nLow Cost\nLow Affordability",      # 区域5
    "Infeasible\nHigh Cost\nLow Affordability",     # 区域6
]
SIX_REGION_COLORS = {
    SIX_REGION_NAMES[0]: '#012A61', # 深蓝 (最理想)
    SIX_REGION_NAMES[1]: '#0B75B3', # 中蓝
    SIX_REGION_NAMES[2]: '#89CAEA', # 浅蓝 (勉强可行)
    SIX_REGION_NAMES[3]: '#F0D2D2', # 浅红 (成本问题)
    SIX_REGION_NAMES[4]: '#DC5654', # 中红 (支付能力问题)
    SIX_REGION_NAMES[5]: '#982B2D', # 深红 (最棘手)
}

# <<<< COPIED 2: 从主代码复制核心分类函数 >>>>
def classify_point(x, y, median_breakeven):
    is_low_cost = x <= median_breakeven
    is_high_affordability_benchmark = y > median_breakeven
    is_feasible = y >= x

    if is_feasible:
        if is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[0]
        elif not is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[1]
        elif is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[2]
        else:
            return SIX_REGION_NAMES[5]
    else: # Infeasible
        if not is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[3]
        elif is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[4]
        elif not is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[5]
        else:
            return SIX_REGION_NAMES[3]

# <<<< MODIFIED 3: 重写整个图例绘制函数 >>>>
def plot_six_region_legend(output_path=None):
    """
    创建一个精确的六区域决策框架图例。
    - 使用 contourf 方法确保与分析逻辑完全一致。
    - 在每个区域内添加标签以增强可读性。
    """
    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)

    # 1. 定义坐标系和分割线
    x_min, x_max = 0, 10
    y_min, y_max = 0, 10
    median_breakeven = 5  # 成本中位线
    
    # 2. 设置坐标轴范围 (关键：翻转X轴)
    ax.set_xlim(x_max, x_min)
    ax.set_ylim(y_min, y_max)
    # ax.set_xlabel('Cost Recovery Electricity Price (LCOE)', fontsize=14, labelpad=10)
    # ax.set_ylabel('Affordable Electricity Price', fontsize=14, labelpad=10)


    # 3. 使用 contourf 精确绘制背景区域
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500), np.linspace(y_min, y_max, 500))
    
    region_to_id = {name: i for i, name in enumerate(SIX_REGION_NAMES)}
    
    vectorized_classify = np.vectorize(classify_point)
    grid_regions = vectorized_classify(xx, yy, median_breakeven)
    
    Z = np.zeros(grid_regions.shape)
    for name, region_id in region_to_id.items():
        Z[grid_regions == name] = region_id
        
    colors = [SIX_REGION_COLORS[name] for name in SIX_REGION_NAMES]
    cmap = matplotlib.colors.ListedColormap(colors)
    
    levels = np.arange(-0.5, len(colors), 1)
    ax.contourf(xx, yy, Z, levels=levels, cmap=cmap, alpha=0.6, zorder=0)

    # 4. 绘制分割线
    # 垂直线：成本中位数
    ax.axvline(x=median_breakeven, color='black', linestyle='--', linewidth=2, zorder=3)
    # 水平线：成本中位数 (作为支付能力基准)
    ax.axhline(y=median_breakeven, color='black', linestyle='--', linewidth=2, zorder=3)
    # 对角线：盈亏平衡线 y=x
    ax.plot([x_max, x_min], [x_max, x_min], color='#be1420', linestyle='--', 
              alpha=1, linewidth=2.5, zorder=3)

    # 6. 清理图表外观
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    # ax.set_xticklabels(['Median\nCost'], fontsize=12)
    # ax.set_yticklabels(['Median\nCost'], fontsize=12)
    # ax.tick_params(axis='both', length=5, width=1)
    
    ax.spines[['right', 'top']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(1)
    
    fig.tight_layout()
    _save_pdf()
    plt.show()

# --- 调用函数以生成图例 ---
if __name__ == '__main__':
    plot_six_region_legend()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- 只加粗Q1-Q6标识符 ---
# 只对Q1-Q6使用加粗，其他文字保持正常
SIX_REGION_NAMES = [
    r"$\mathbf{Q1:}$" + "\n" + "Feasible" + "\n" + "Low Cost" + "\n" + "High Affordability",
    r"$\mathbf{Q2:}$" + "\n" + "Feasible" + "\n" + "High Cost" + "\n" + "High Affordability",
    r"$\mathbf{Q3:}$" + "\n" + "Feasible" + "\n" + "Low Cost" + "\n" + "Low Affordability",
    r"$\mathbf{Q4:}$" + "\n" + "Infeasible" + "\n" + "High Cost" + "\n" + "High Affordability",
    r"$\mathbf{Q5:}$" + "\n" + "Infeasible" + "\n" + "Low Cost" + "\n" + "Low Affordability",
    r"$\mathbf{Q6:}$" + "\n" + "Infeasible" + "\n" + "High Cost" + "\n" + "Low Affordability",
]

SIX_REGION_COLORS_LIST = ['#012A61', '#0B75B3', '#89CAEA', '#F0D2D2', '#DC5654', '#982B2D']


def create_seven_region_legend():
    # 创建图例补丁对象，每个对象包含颜色和标签
    legend_patches = [mpatches.Patch(color=color, label=name, alpha=1, edgecolor='none')
                      for name, color in zip(SIX_REGION_NAMES, SIX_REGION_COLORS_LIST)]

    # 重新排列为2行3列布局
    reordered_patches_np = np.array(legend_patches).reshape(2, 3).T.flatten()

    # 创建图形和图例
    fig = plt.figure(figsize=(12, 3), dpi=300)  # 设置图形尺寸和分辨率

    legend = fig.legend(
        handles=reordered_patches_np.tolist(),
        # handles=legend_patches,
        loc='center',           # 图例位置居中
        frameon=False,          # 不显示边框
        title='Quadrant Chart', # 图例标题
        title_fontsize=18,      # 标题字体大小
        fontsize=14,            # 标签字体大小
        ncol=3                  # 3列布局
    )

    fig.gca().set_axis_off()    # 隐藏坐标轴
    
    # 显示图形
    _save_pdf()
    plt.show()
    
    # 可选：保存图形文件
    # plt.savefig("legend_bold.png", bbox_inches='tight', pad_inches=0.1, transparent=True)
    # plt.close()


create_seven_region_legend()

# Vertical (single-column) companion to the horizontal quadrant legend above
def create_quadrant_legend_vertical():
    patches_v = [mpatches.Patch(color=color, label=name, alpha=1, edgecolor='none')
                 for name, color in zip(SIX_REGION_NAMES, SIX_REGION_COLORS_LIST)]
    figv = plt.figure(figsize=(3.5, 9), dpi=300)
    figv.legend(handles=patches_v, loc='center', frameon=False, fontsize=14, ncol=1)
    figv.gca().set_axis_off()
    figv.savefig('pdf/fig2_1_quadrant_legend_vertical.pdf', bbox_inches='tight')
    figv.savefig('svg/fig2_1_quadrant_legend_vertical.svg', bbox_inches='tight')
    figv.savefig('png/fig2_1_quadrant_legend_vertical.png', dpi=300, bbox_inches='tight')
    plt.close(figv)

create_quadrant_legend_vertical()


In [ ]:
# --- 导入必要的库 ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import geopandas as gpd

# <<<< MODIFIED 1: 更新为六区域的定义和颜色方案 >>>>
SIX_REGION_NAMES = [
    "Feasible\nLow Cost\nHigh Affordability",       # 区域1
    "Feasible\nHigh Cost\nHigh Affordability",      # 区域2
    "Feasible\nLow Cost\nLow Affordability",        # 区域3
    "Infeasible\nHigh Cost\nHigh Affordability",    # 区域4
    "Infeasible\nLow Cost\nLow Affordability",      # 区域5
    "Infeasible\nHigh Cost\nLow Affordability",     # 区域6
]
SIX_REGION_COLORS = {
    SIX_REGION_NAMES[0]: '#012A61', # 深蓝 (最理想)
    SIX_REGION_NAMES[1]: '#0B75B3', # 中蓝
    SIX_REGION_NAMES[2]: '#89CAEA', # 浅蓝 (勉强可行)
    SIX_REGION_NAMES[3]: '#EE9D9F', # 浅红 (成本问题)
    SIX_REGION_NAMES[4]: '#DC5654', # 中红 (支付能力问题)
    SIX_REGION_NAMES[5]: '#982B2D', # 深红 (最棘手)
}

# --- 全局中位线定义 ---
# <<<< MODIFIED 2: 移除不再需要的可负担能力中位数全局变量 >>>>
GLOBAL_MEDIAN_BREAKEVEN = None

def initialize_global_medians(df):
    """根据基准情景 (output_0) 初始化全局成本中位数"""
    global GLOBAL_MEDIAN_BREAKEVEN
    df_baseline = df[df['scenario'] == 'output_0'].copy()
    if not df_baseline.empty:
        GLOBAL_MEDIAN_BREAKEVEN = df_baseline['tariff_breakeven'].median()
        print(f"全局中位线已设置 (基于 'output_0' 场景):")
        print(f"   - 成本回收电价中位数: {GLOBAL_MEDIAN_BREAKEVEN:.3f} USD/kWh")
    else:
        print("警告：未找到 'output_0' 场景数据，无法设置全局中位线！")

# --- 核心分类函数 ---
# <<<< MODIFIED 3: 重写核心分类函数以适配六区域逻辑 >>>>
def classify_point(x, y, median_breakeven):
    """
    根据给定的成本中位数对单个数据点 (x, y) 进行分类。
    x: tariff_breakeven, y: tariff_affordable
    """
    is_low_cost = x <= median_breakeven
    is_high_affordability_benchmark = y > median_breakeven # 支付能力与成本中位数比较
    is_feasible = y >= x

    if is_feasible:
        if is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[0]
        elif not is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[1]
        elif is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[2]
        else:
            return SIX_REGION_NAMES[5]
    else: # Infeasible
        if not is_low_cost and is_high_affordability_benchmark:
            return SIX_REGION_NAMES[3]
        elif is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[4]
        elif not is_low_cost and not is_high_affordability_benchmark:
            return SIX_REGION_NAMES[5]
        else:
            return SIX_REGION_NAMES[3]

def classify_six_regions_dataframe(df_scenario, median_breakeven): # <<<< MODIFIED: 函数名和参数
    """对 DataFrame 中的每一行数据应用六区域分类法"""
    if median_breakeven is None:
        raise ValueError("全局中位线未初始化。请先调用 initialize_global_medians()。")
    
    df_scenario['region'] = df_scenario.apply(
        lambda row: classify_point(row['tariff_breakeven'], row['tariff_affordable'], median_breakeven), # <<<< MODIFIED: 调用新的分类函数
        axis=1
    )
    return df_scenario

# --- Nature 风格参数设置 ---
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 20,
    'axes.labelsize': 20,
    'axes.titlesize': 20,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'figure.dpi': 300,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

# --- 地图样式参数 ---
OCEAN_COLOR = "#ffffff"  # 海洋颜色 - 白色
LAND_COLOR = '#cecece'   # 陆地颜色 - 浅灰色
BORDERS_COLOR = '#a0a0a0'  # 边界线颜色 - 深灰色
GRIDLINE_COLOR = '#cccccc'  # 网格线颜色 - 浅灰色

# --- 散点图样式参数 ---
POINT_SIZE = 48  # 散点大小
POINT_ALPHA = 0.85  # 散点透明度
POINT_EDGECOLOR = '#333333'  # 散点边框颜色
POINT_LINEWIDTH = 0.25  # 散点边框宽度

# --- 数据加载与预处理 ---
print("正在加载数据...")
ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")

data_path = Path("../result/island_viability_summary_electric.csv")  # 修正数据路径
if not data_path.exists():
    print(f"错误：数据文件不存在 - {data_path}")
    raise FileNotFoundError(f"数据文件不存在: {data_path}")

df_global = pd.read_csv(data_path)
print(f"数据加载成功，共 {len(df_global)} 条记录。")

# 初始化全局中位数
initialize_global_medians(df_global)

# --- 全球地图可视化 ---
# <<<< MODIFIED 5: 更新主绘图函数以反映六区域模型 >>>>
def plot_global_island_map_six_regions(df_data, scenario_title):
    """绘制全球岛屿六区域分类分布图"""
    fig = plt.figure(figsize=(15, 8.5))  # 设置图形大小
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())  # 使用Robinson投影

    ax.set_facecolor(OCEAN_COLOR)  # 设置背景为海洋颜色
    ax.add_feature(cfeature.LAND, color=LAND_COLOR, zorder=0, alpha=0.3)  # 添加陆地特征
    ax.add_feature(cfeature.OCEAN, color=OCEAN_COLOR, zorder=0, alpha=0.3)  # 添加海洋特征
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=2, alpha=0.6)  # 添加海岸线
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color=BORDERS_COLOR, alpha=0.7, zorder=2)  # 添加国界线
    ax.set_global()  # 设置全球视图
    # --- Longitude/latitude graticule (Nature Comms checklist) ---
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linewidth=0.4, color='gray', alpha=0.5, linestyle='--',
                      xlocs=range(-180, 181, 60), ylocs=range(-60, 91, 30))
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 16, 'color': '#404040'}
    gl.ylabel_style = {'size': 16, 'color': '#404040'}
    ax.spines['geo'].set_visible(False)  # 隐藏地理边框
    
    if ipcc_regions is not None and not ipcc_regions.empty:
        # 遍历每个IPCC区域
        for _, region_row in ipcc_regions.iterrows():
            # 绘制区域轮廓线
            ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                              facecolor='none',      # 设置为'none'以只显示边框
                              edgecolor='#555555',     # 边框颜色
                              linewidth=0.8,         # 边框线宽
                              linestyle='--',         # 虚线样式
                              zorder=1)              # zorder确保它在密度图之上
                    
    print(f"\n正在绘制场景 {scenario_title} 的全球地图...")
    # 按预设顺序绘制各区域的散点
    for region_name in SIX_REGION_NAMES:
        color = SIX_REGION_COLORS.get(region_name)
        if color is None:
            continue
        
        region_data = df_data[df_data['region'] == region_name]
        if not region_data.empty:
            ax.scatter(
                region_data['lon'], region_data['lat'],  # 经纬度数据
                c=color,  # 散点颜色
                s=POINT_SIZE,  # 散点大小
                alpha=POINT_ALPHA,  # 透明度
                transform=ccrs.PlateCarree(),  # 坐标变换
                edgecolors=POINT_EDGECOLOR,  # 边框颜色
                linewidth=POINT_LINEWIDTH,  # 边框宽度
                label=region_name,  # 图例标签
                zorder=3  # 图层顺序
            )

    plt.tight_layout(pad=1.0)  # 调整布局
    _save_pdf()
    plt.show()  # 显示图形

# --- 独立图例生成 ---
# <<<< MODIFIED 6: 更新图例生成函数以反映六区域模型 >>>>
def create_six_region_legend():
    """为六区域地图创建一个独立的、格式化的图例"""
    fig_legend = plt.figure(figsize=(12, 3))  # 调整尺寸以适应6个图例项
    
    legend_elements = []
    for region_name, color in SIX_REGION_COLORS.items():
        legend_elements.append(
            plt.Line2D([0], [0], marker='o', color='w',
                       markerfacecolor=color,  # 散点颜色
                       markersize=18,  # 散点大小
                       markeredgecolor=POINT_EDGECOLOR,  # 边框颜色
                       markeredgewidth=0.5,  # 边框宽度
                       label=region_name.replace('\n', ' '))  # 将标签中的换行符替换为空格
        )
    
    fig_legend.legend(
        handles=legend_elements,
        loc='center',  # 图例位置居中
        frameon=False,  # 不显示边框
        fontsize=16,   # 调整字号
        ncol=3,        # 6个项目，3列布局更美观
        columnspacing=2.0,  # 列间距
        handletextpad=0.8   # 图例标记和文本间距
    )
    
    fig_legend.gca().set_axis_off()  # 隐藏坐标轴
    plt.tight_layout()  # 调整布局
    _save_pdf()
    plt.show()  # 显示图形

# --- 最后单独生成一次图例 ---
print("\n--- 生成独立的图例 ---")
create_six_region_legend()

In [ ]:
# 选择要分析的场景 - 修改为循环处理所有场景
available_scenarios = df_global['scenario'].unique()
print(f"\n可用场景: {available_scenarios}")
print(f"总共 {len(available_scenarios)} 个场景")

# <<<< MODIFIED 4: 循环处理所有场景 >>>>
# 循环处理每个场景
for scenario_name in available_scenarios:
    print(f"\n{'='*60}")
    print(f"正在处理场景: {scenario_name}")
    print(f"{'='*60}")

    df_scenario = df_global[df_global['scenario'] == scenario_name].copy()
    print(f"当前场景包含 {len(df_scenario)} 个岛屿数据。")

    # 应用六区域分类逻辑
    if GLOBAL_MEDIAN_BREAKEVEN is not None:
        print("正在对岛屿进行六区域分类...")
        df_scenario = classify_six_regions_dataframe(df_scenario, GLOBAL_MEDIAN_BREAKEVEN)
        print("分类完成。")
        print(f"区域分布情况:\n{df_scenario['region'].value_counts()}")

        # 为当前场景生成地图
        if not df_scenario.empty:
            print(f"\n正在为场景 '{scenario_name}' 生成地图...")
            plot_global_island_map_six_regions(df_scenario, scenario_name)
        else:
            print(f"警告：场景 '{scenario_name}' 没有可用数据，跳过地图生成。")

    else:
        print("错误：无法进行分类，因为全局中位线未成功设置。")
        break

# --- 总的可行与不可行岛屿饼状图 ---

import matplotlib.pyplot as plt
import numpy as np

def create_feasibility_pie_chart():
    """
    创建显示总的可行岛屿和不可行岛屿比例的饼状图
    基于 viability_gap 的正负值来判断岛屿的可行性
    """
    
    # 选择基线scenario (output_0) 进行分析
    baseline_scenario = 'output_0'
    df_baseline = df_global[df_global['scenario'] == baseline_scenario].copy()
    
    print(f"正在分析scenario: {baseline_scenario}")
    print(f"总岛屿数量: {len(df_baseline)}")
    
    # 根据viability_gap判断可行性
    # viability_gap = tariff_breakeven - tariff_affordable
    # viability_gap > 0: 不可行 (需要的电价高于能承受的)
    # viability_gap <= 0: 可行 (需要的电价低于或等于能承受的)
    
    feasible_islands = len(df_baseline[df_baseline['viability_gap'] <= 0])
    infeasible_islands = len(df_baseline[df_baseline['viability_gap'] > 0])
    total_islands = len(df_baseline)
    
    # 计算百分比
    feasible_percentage = (feasible_islands / total_islands) * 100
    infeasible_percentage = (infeasible_islands / total_islands) * 100
    
    print(f"可行岛屿: {feasible_islands} ({feasible_percentage:.1f}%)")
    print(f"不可行岛屿: {infeasible_islands} ({infeasible_percentage:.1f}%)")
    
    # 设置饼状图数据
    sizes = [feasible_islands, infeasible_islands]
    labels = ['Feasible Islands', 'Infeasible Islands']
    colors = ['#4C72B0', '#C44E52']  # 蓝色表示可行，红色表示不可行
    explode = (0.05, 0.05)  # 稍微分离两个部分
    
    # 创建饼状图
    fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
    
    # 绘制饼状图
    wedges = ax.pie(sizes, 
                                    #   labels=labels,
                                      colors=colors,
                                      explode=explode,
                                    #   autopct='%1.1f%%',
                                      startangle=90,
                                      textprops={'fontsize': 20, 'fontweight': 'bold'},
                                      wedgeprops={'edgecolor': 'white', 'linewidth': 2}
                                    )
    
    # 美化百分比文字
    # for autotext in autotexts:
    #     autotext.set_color('white')
    #     autotext.set_fontsize(24)
    #     autotext.set_fontweight('bold')
    
    # # 美化标签文字
    # for text in texts:
    #     text.set_fontsize(22)
    #     text.set_fontweight('bold')
    #     text.set_color('#333333')
    
    # 添加总数标注
    # ax.text(0, -1.3, f'Total Islands: {total_islands:,}', 
    #         ha='center', va='center', 
    #         fontsize=18, fontweight='bold',
    #         bbox=dict(boxstyle="round,pad=0.3", facecolor='lightgray', alpha=0.7))
    
    # 添加scenario标注
    # ax.text(0, 1.3, f'Scenario: {baseline_scenario.replace("output_", "").title()}', 
    #         ha='center', va='center', 
    #         fontsize=20, fontweight='bold',
    #         color='#333333')
    
    # 确保饼状图是圆形
    ax.set_aspect('equal')
    
    # 移除坐标轴
    ax.axis('off')
    
    plt.tight_layout()
    _save_pdf()
    plt.show()
    
    # 创建详细统计表
    print(f"\n=== 岛屿可行性统计汇总 ===")
    print(f"Scenario: {baseline_scenario}")
    print(f"{'Category':<20} {'Count':<10} {'Percentage':<12}")
    print("-" * 42)
    print(f"{'Feasible':<20} {feasible_islands:<10,} {feasible_percentage:<12.1f}%")
    print(f"{'Infeasible':<20} {infeasible_islands:<10,} {infeasible_percentage:<12.1f}%")
    print("-" * 42)
    print(f"{'Total':<20} {total_islands:<10,} {'100.0':<12}%")
    
    return feasible_islands, infeasible_islands, total_islands

# 执行饼状图绘制
print("=== 创建岛屿可行性饼状图 ===")
if 'df_global' in locals():
    # 单一scenario饼状图
    feasible_count, infeasible_count, total_count = create_feasibility_pie_chart()


else:
    print("错误: 未找到df_global数据，请确保之前的cell已经运行")

In [ ]:
# === 比较分析：识别情景间分类变化的岛屿 ===

import matplotlib.pyplot as plt
import numpy as np

# 选择要比较的情景
baseline_scenario = 'output_0'     # 基准情景
compare_scenario_1 = 'output_2020'  # 2020情景
compare_scenario_2 = 'output_2050'  # 2050情景

def get_scenario_classifications(df, scenario_name, median_breakeven):
    """获取特定情景下所有岛屿的分类结果"""
    df_scenario = df[df['scenario'] == scenario_name].copy()
    if not df_scenario.empty and median_breakeven is not None:
        df_scenario = classify_six_regions_dataframe(df_scenario, median_breakeven)
        # 创建岛屿ID到分类的映射（使用lat,lon作为唯一标识）
        classification_map = {}
        for _, row in df_scenario.iterrows():
            island_id = (row['lat'], row['lon'])  # 使用坐标作为唯一标识
            classification_map[island_id] = row['region']
        return df_scenario, classification_map
    else:
        return pd.DataFrame(), {}

def identify_classification_changes(baseline_map, compare_map):
    """识别两个情景间发生分类变化的岛屿"""
    changed_islands = []
    
    for island_id in baseline_map.keys():
        if island_id in compare_map:
            baseline_class = baseline_map[island_id]
            compare_class = compare_map[island_id]
            
            if baseline_class != compare_class:
                changed_islands.append({
                    'island_id': island_id,
                    'lat': island_id[0],
                    'lon': island_id[1], 
                    'baseline_class': baseline_class,
                    'compare_class': compare_class
                })
    
    return changed_islands

print("=== 开始比较分析 ===")

# 获取各情景的分类结果
print(f"正在分析基准情景: {baseline_scenario}")
df_baseline, baseline_classifications = get_scenario_classifications(df_global, baseline_scenario, GLOBAL_MEDIAN_BREAKEVEN)

print(f"正在分析比较情景1: {compare_scenario_1}")  
df_compare_1, compare_1_classifications = get_scenario_classifications(df_global, compare_scenario_1, GLOBAL_MEDIAN_BREAKEVEN)

print(f"正在分析比较情景2: {compare_scenario_2}")
df_compare_2, compare_2_classifications = get_scenario_classifications(df_global, compare_scenario_2, GLOBAL_MEDIAN_BREAKEVEN)

# 识别发生变化的岛屿
print(f"\n正在识别 {baseline_scenario} 与 {compare_scenario_1} 间的分类变化...")
changed_islands_2020 = identify_classification_changes(baseline_classifications, compare_1_classifications)

print(f"找到 {len(changed_islands_2020)} 个岛屿在两个情景间发生了分类变化")

if len(changed_islands_2020) > 0:
    # 统计变化类型
    change_stats = {}
    for island in changed_islands_2020:
        change_key = f"{island['baseline_class']} → {island['compare_class']}"
        if change_key not in change_stats:
            change_stats[change_key] = 0
        change_stats[change_key] += 1
    
    print(f"\n=== 分类变化统计 ({baseline_scenario} → {compare_scenario_1}) ===")
    for change_type, count in sorted(change_stats.items(), key=lambda x: x[1], reverse=True):
        print(f"{change_type}: {count} 个岛屿")
        
    # 创建变化岛屿的数据框，用于地图绘制
    changed_islands_df = pd.DataFrame(changed_islands_2020)
    
    # 为这些变化的岛屿获取三个情景的完整数据
    changed_island_coords = [(island['lat'], island['lon']) for island in changed_islands_2020]
    
    # 筛选出发生变化的岛屿在各情景下的数据
    def filter_changed_islands(df_scenario, changed_coords):
        """筛选出发生变化的岛屿的数据"""
        filtered_data = []
        for _, row in df_scenario.iterrows():
            island_coord = (row['lat'], row['lon'])
            if island_coord in changed_coords:
                filtered_data.append(row)
        return pd.DataFrame(filtered_data) if filtered_data else pd.DataFrame()
    
    df_baseline_filtered = filter_changed_islands(df_baseline, changed_island_coords)
    df_2020_filtered = filter_changed_islands(df_compare_1, changed_island_coords)  
    df_2050_filtered = filter_changed_islands(df_compare_2, changed_island_coords)
    
    print(f"\n筛选后的数据:")
    print(f"基准情景变化岛屿数: {len(df_baseline_filtered)}")
    print(f"2020情景变化岛屿数: {len(df_2020_filtered)}")
    print(f"2050情景变化岛屿数: {len(df_2050_filtered)}")
else:
    print("未找到发生分类变化的岛屿")

In [ ]:
# === 为发生变化的岛屿绘制三个情景的对比地图 ===

def plot_three_scenario_comparison_maps(df_baseline, df_2020, df_2050, scenario_names):
    """
    绘制三个情景下发生变化岛屿的对比地图 - 竖向排列
    """
    # 创建3行1列的子图布局（竖向排列）
    fig, axes = plt.subplots(3, 1, figsize=(15, 18),  # 设置图形大小 - 调整为竖向比例
                            subplot_kw={'projection': ccrs.Robinson()},  # 使用Robinson投影
                            dpi=300)  # 设置分辨率
    
    # 要绘制的数据和标题
    scenario_data = [
        (df_baseline, scenario_names[0], axes[0]),
        (df_2020, scenario_names[1], axes[1]), 
        (df_2050, scenario_names[2], axes[2])
    ]
    
    for df_data, scenario_name, ax in scenario_data:
        # 设置地图基本特征
        ax.set_facecolor(OCEAN_COLOR)  # 设置背景为海洋颜色
        ax.add_feature(cfeature.LAND, color=LAND_COLOR, zorder=0, alpha=0.3)  # 添加陆地特征
        ax.add_feature(cfeature.OCEAN, color=OCEAN_COLOR, zorder=0, alpha=0.3)  # 添加海洋特征  
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=2, alpha=0.6)  # 添加海岸线
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, 
                      color=BORDERS_COLOR, alpha=0.7, zorder=2)  # 添加国界线
        ax.set_global()  # 设置全球视图
        # --- Longitude/latitude graticule (Nature Comms checklist) ---
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                          linewidth=0.4, color='gray', alpha=0.5, linestyle='--',
                          xlocs=range(-180, 181, 60), ylocs=range(-60, 91, 30))
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 16, 'color': '#404040'}
        gl.ylabel_style = {'size': 16, 'color': '#404040'}
        ax.spines['geo'].set_visible(False)  # 隐藏地理边框
        
        # 添加IPCC区域轮廓（可选）
        if ipcc_regions is not None and not ipcc_regions.empty:
            for _, region_row in ipcc_regions.iterrows():
                ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                                 facecolor='none',      # 设置为'none'以只显示边框
                                 edgecolor='#555555',   # 边框颜色
                                 linewidth=0.6,        # 边框线宽
                                 linestyle='--',        # 虚线样式
                                 alpha=0.4,            # 透明度
                                 zorder=1)             # 图层顺序
        
        # 绘制各区域的散点
        if not df_data.empty:
            print(f"正在绘制 {scenario_name} 情景，包含 {len(df_data)} 个变化岛屿...")
            
            for region_name in SIX_REGION_NAMES:
                color = SIX_REGION_COLORS.get(region_name)
                if color is None:
                    continue
                
                region_data = df_data[df_data['region'] == region_name]
                if not region_data.empty:
                    ax.scatter(
                        region_data['lon'], region_data['lat'],  # 经纬度数据
                        c=color,  # 散点颜色
                        s=POINT_SIZE * 1.2,  # 散点大小（稍大一些以突出变化岛屿）
                        alpha=POINT_ALPHA,  # 透明度
                        transform=ccrs.PlateCarree(),  # 坐标变换
                        edgecolors=POINT_EDGECOLOR,  # 边框颜色
                        linewidth=POINT_LINEWIDTH * 2,  # 边框宽度（稍粗一些）
                        label=region_name,  # 图例标签
                        zorder=3  # 图层顺序
                    )
        else:
            print(f"警告：{scenario_name} 情景没有变化岛屿数据")
        
        # 设置子图标题（可选，根据需要开启）
        # ax.set_title(scenario_name, fontsize=16, pad=10)
    
    plt.tight_layout(pad=1.0)  # 调整布局
    _save_pdf()
    plt.show()  # 显示图形

# 执行三情景对比地图绘制
if len(changed_islands_2020) > 0:
    print("\n=== 开始绘制三情景对比地图（竖向排列）===")
    
    scenario_titles = [
        f"Baseline ({baseline_scenario})",
        f"2020 Scenario ({compare_scenario_1})", 
        f"2050 Scenario ({compare_scenario_2})"
    ]
    
    # 检查数据完整性
    print(f"\n数据检查:")
    print(f"基准情景数据: {len(df_baseline_filtered)} 个岛屿")
    print(f"2020情景数据: {len(df_2020_filtered)} 个岛屿") 
    print(f"2050情景数据: {len(df_2050_filtered)} 个岛屿")
    
    # 确保所有数据都有区域分类
    if len(df_baseline_filtered) > 0 and len(df_2020_filtered) > 0 and len(df_2050_filtered) > 0:
        plot_three_scenario_comparison_maps(
            df_baseline_filtered, 
            df_2020_filtered, 
            df_2050_filtered,
            scenario_titles
        )
        
        # 生成变化统计汇总
        print(f"\n=== 变化岛屿区域分布统计 ===")
        
        for i, (df_data, scenario_name) in enumerate([
            (df_baseline_filtered, scenario_titles[0]),
            (df_2020_filtered, scenario_titles[1]), 
            (df_2050_filtered, scenario_titles[2])
        ]):
            print(f"\n{scenario_name}:")
            if not df_data.empty and 'region' in df_data.columns:
                region_counts = df_data['region'].value_counts()
                for region, count in region_counts.items():
                    print(f"  {region.replace(chr(10), ' ')}: {count} 个岛屿")
            else:
                print("  无数据或缺少区域分类")
    else:
        print("警告：部分情景数据缺失，无法生成对比地图")
else:
    print("未找到发生分类变化的岛屿，无法生成对比地图")

In [ ]:
# ============================================================
# 全球岛屿“可行性缺口”地图 (Feasibility Gap map)
#   散点: 每个岛屿的可行性缺口 (USD kWh$^{-1}$) = 可负担电价 - 成本回收电价
#         = -viability_gap  (正值=可行/蓝, 负值=不可行/红)
#   区域填色: 各 IPCC 区域内“不可行岛屿占比 (%)”(青色 choropleth)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

plt.rcParams['font.family'] = 'Arial'

# ---------- 参数 ----------
SCENARIO = 'output_0'   # 基准情景
GAP_LIM  = 0.4          # 可行性缺口色带范围 ±0.4 (超出部分用箭头表示)

# ---------- 加载岛屿可行性数据 ----------
_df = pd.read_csv(Path("../result/island_viability_summary_electric.csv"))
_base = (_df[_df['scenario'] == SCENARIO]
         .dropna(subset=['viability_gap'])
         .drop_duplicates('island_id')
         .copy())
# 可行性缺口 = 可负担电价 - 成本回收电价 = -viability_gap
_base['feasibility_gap'] = -_base['viability_gap']
_base['infeasible'] = (_base['feasibility_gap'] < 0).astype(int)
print(f"基准情景 {SCENARIO}: {len(_base)} 个岛屿, 整体不可行比例 {_base['infeasible'].mean()*100:.1f}%")

# ---------- 各 IPCC 区域不可行比例 ----------
_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson").to_crs("EPSG:4326")
_IDCOL = 'Acronym'
_gdf = gpd.GeoDataFrame(
    _base, geometry=[Point(xy) for xy in zip(_base['lon'], _base['lat'])], crs="EPSG:4326")
_joined = gpd.sjoin(_gdf, _regions[[_IDCOL, 'geometry']], how='left', predicate='within')
_grp = _joined.groupby(_IDCOL).agg(n=('island_id', 'size'), inf=('infeasible', 'sum'))
_grp['pct'] = _grp['inf'] / _grp['n'] * 100.0
_region_pct = _grp['pct'].to_dict()

# ---------- 颜色映射 ----------
gap_cmap = mpl.colors.LinearSegmentedColormap.from_list(  # 红(不可行)→黄→蓝(可行), 自定义配色
    'gap_rdbu', ['#d73221', '#e35235', '#e48070', '#fcb777', '#fde699',
                 '#fef4ae', '#d2edf2', '#6491c1'])
gap_norm = mpl.colors.Normalize(vmin=-GAP_LIM, vmax=GAP_LIM)
teal_cmap = mpl.colors.LinearSegmentedColormap.from_list(
    'infeasible_teal', ['#ffffff', '#cfe3e3', '#9fc8c8', '#5fa3a3', '#2f7a7a'])
pct_norm = mpl.colors.Normalize(vmin=0, vmax=100)

# ---------- 画布 ----------
fig = plt.figure(figsize=(15, 8.5), dpi=300)
ax    = fig.add_axes([0.02, 0.14, 0.84, 0.84], projection=ccrs.Robinson())  # 主地图
cax_v = fig.add_axes([0.885, 0.30, 0.018, 0.52])    # 右侧竖直色带 (可行性缺口)
cax_h = fig.add_axes([0.20, 0.07, 0.46, 0.022])     # 底部水平色带 (区域不可行%)

ax.set_facecolor('#ffffff')
ax.add_feature(cfeature.LAND,      color='#cecece', zorder=0, alpha=0.35)
ax.add_feature(cfeature.OCEAN,     color='#ffffff', zorder=0, alpha=0.30)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=2, alpha=0.6)
ax.add_feature(cfeature.BORDERS,   linestyle=':', linewidth=0.4, color='#a0a0a0', alpha=0.7, zorder=2)
ax.set_global()

gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=0.4, color='gray', alpha=0.5, linestyle='--',
                  xlocs=range(-180, 181, 60), ylocs=range(-60, 91, 30))
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 16, 'color': '#404040'}
gl.ylabel_style = {'size': 16, 'color': '#404040'}
ax.spines['geo'].set_visible(False)

# ---------- 区域填色 (不可行比例) ----------
for _, _row in _regions.iterrows():
    _pct = _region_pct.get(_row[_IDCOL], None)
    _face = teal_cmap(pct_norm(_pct)) if _pct is not None else 'none'
    ax.add_geometries([_row['geometry']], crs=ccrs.PlateCarree(),
                      facecolor=_face, edgecolor='#7a7a7a',
                      linewidth=0.5, linestyle='--', alpha=0.55, zorder=1)

# ---------- 散点 (可行性缺口); 不可行点画在上层以突出 ----------
_bs = _base.sort_values('feasibility_gap', ascending=False)
sc = ax.scatter(_bs['lon'], _bs['lat'],
                c=_bs['feasibility_gap'], cmap=gap_cmap, norm=gap_norm,
                s=42, alpha=0.9, transform=ccrs.PlateCarree(),
                edgecolors='#333333', linewidth=0.25, zorder=3)

# ---------- 竖直色带: 可行性缺口 ----------
cb_v = fig.colorbar(mpl.cm.ScalarMappable(norm=gap_norm, cmap=gap_cmap),
                    cax=cax_v, orientation='vertical', extend='both')
cb_v.set_label('Feasibility Gap (USD kWh$^{-1}$)', fontsize=26)
cb_v.set_ticks([-0.4, -0.2, 0.0, 0.2, 0.4])
cb_v.ax.tick_params(labelsize=20)

# ---------- 水平色带: 区域不可行比例 ----------
cb_h = fig.colorbar(mpl.cm.ScalarMappable(norm=pct_norm, cmap=teal_cmap),
                    cax=cax_h, orientation='horizontal')
cb_h.set_label('Infeasible Islands in IPCC Regions (%)', fontsize=26)
cb_h.set_ticks([0, 20, 40, 60, 80, 100])
cb_h.ax.tick_params(labelsize=20)

_save_pdf()
plt.show()
